In [5]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np

# Imports from PyTorch.
import torch
from torch import nn
from torchvision import datasets, transforms

import sys
# get the path of the current file
file = os.path.abspath('')
path = os.path.join(file, '../../src')
sys.path.append(path)
print(path)


# Imports from aihwkit.
from aihwkit.nn import AnalogConv2d, AnalogLinear, AnalogSequential
from aihwkit.optim import AnalogSGD
from aihwkit.simulator.configs import (
    SingleRPUConfig,
    FloatingPointRPUConfig,
    ConstantStepDevice,
    FloatingPointDevice,
)

from aihwkit.simulator.parameters import (
    WeightQuantizerParameter,
)

from aihwkit.simulator.rpu_base import cuda



# Check device
USE_CUDA = 0
if cuda.is_compiled():
    USE_CUDA = 1

if USE_CUDA:
    torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0" if USE_CUDA else "cpu")
print("Device: ", DEVICE)


cur_dir = os.getcwd()
print("Current directory: ", cur_dir)


# Path to store datasets
PATH_DATASET = cur_dir + "/../data/mnist"


# Training parameters
SEED = 1
N_EPOCHS = 30
BATCH_SIZE = 64
LEARNING_RATE = 0.01
N_CLASSES = 10

/home/ecabiati/cellar/aihwkit/sandbox/cuda/../../src
Device:  cuda:0
Current directory:  /home/ecabiati/cellar/aihwkit/sandbox/cuda


In [6]:



# Select the device model to use in the training.
# * If `SingleRPUConfig(device=ConstantStepDevice())` then analog tiles with
#   constant step devices will be used,
# * If `FloatingPointRPUConfig(device=FloatingPointDevice())` then standard
#   floating point devices will be used
USE_ANALOG_TRAINING = False



def load_images():
    """Load images for train from torchvision datasets."""

    transform = transforms.Compose([transforms.ToTensor()])
    train_set = datasets.MNIST(PATH_DATASET, download=True, train=True, transform=transform)
    val_set = datasets.MNIST(PATH_DATASET, download=True, train=False, transform=transform)
    train_data = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    validation_data = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    return train_data, validation_data


def create_analog_network(RPU_CONFIG):
    """Return a LeNet5 inspired analog model."""
    channel = [16, 32, 512, 128]
    model = AnalogSequential(
        AnalogConv2d(
            in_channels=1, out_channels=channel[0], kernel_size=5, stride=1, rpu_config=RPU_CONFIG
        ),
        nn.Tanh(),
        nn.MaxPool2d(kernel_size=2),
        AnalogConv2d(
            in_channels=channel[0],
            out_channels=channel[1],
            kernel_size=5,
            stride=1,
            rpu_config=RPU_CONFIG,
        ),
        nn.Tanh(),
        nn.MaxPool2d(kernel_size=2),
        nn.Tanh(),
        nn.Flatten(),
        AnalogLinear(in_features=channel[2], out_features=channel[3], rpu_config=RPU_CONFIG),
        nn.Tanh(),
        AnalogLinear(in_features=channel[3], out_features=N_CLASSES, rpu_config=RPU_CONFIG),
        nn.LogSoftmax(dim=1),
    )

    return model


def create_sgd_optimizer(model, learning_rate):
    """Create the analog-aware optimizer.

    Args:
        model (nn.Module): model to be trained
        learning_rate (float): global parameter to define learning rate

    Returns:
        nn.Module: Analog optimizer
    """
    optimizer = AnalogSGD(model.parameters(), lr=learning_rate)
    optimizer.regroup_param_groups(model)

    return optimizer


def train_step(train_data, model, criterion, optimizer):
    """Train network.

    Args:
        train_data (DataLoader): Validation set to perform the evaluation
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss
        optimizer (Optimizer): analog model optimizer

    Returns:
        nn.Module, nn.Module, float:  model, optimizer and loss for per epoch
    """
    total_loss = 0

    model.train()

    for images, labels in train_data:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()

        # Add training Tensor to the model (input).
        output = model(images)
        loss = criterion(output, labels)

        # Run training (backward propagation).
        loss.backward()

        # Optimize weights.
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    epoch_loss = total_loss / len(train_data.dataset)

    return model, optimizer, epoch_loss


def test_evaluation(validation_data, model, criterion):
    """Test trained network.

    Args:
        validation_data (DataLoader): Validation set to perform the evaluation
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss

    Returns:
        nn.Module, float, float, float:  model, loss, error, and accuracy
    """
    total_loss = 0
    predicted_ok = 0
    total_images = 0

    model.eval()

    for images, labels in validation_data:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        pred = model(images)
        loss = criterion(pred, labels)
        total_loss += loss.item() * images.size(0)

        _, predicted = torch.max(pred.data, 1)
        total_images += labels.size(0)
        predicted_ok += (predicted == labels).sum().item()
        accuracy = predicted_ok / total_images * 100
        error = (1 - predicted_ok / total_images) * 100

    epoch_loss = total_loss / len(validation_data.dataset)

    return model, epoch_loss, error, accuracy


def training_loop(model, criterion, optimizer, train_data, validation_data, epochs, RESULTS, model_name, print_every=1):
    """Training loop.

    Args:
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss
        optimizer (Optimizer): analog model optimizer
        train_data (DataLoader): Validation set to perform the evaluation
        validation_data (DataLoader): Validation set to perform the evaluation
        epochs (int): global parameter to define epochs number
        print_every (int): defines how many times to print training progress

    Returns:
        nn.Module, Optimizer, Tuple: model, optimizer,
            and a tuple of train losses, validation losses, and test
            error
    """
    train_losses = []
    valid_losses = []
    test_error = []
    best_accuracy = 0

    # Train model
    for epoch in range(0, epochs):
        # Train_step
        model, optimizer, train_loss = train_step(train_data, model, criterion, optimizer)
        train_losses.append(train_loss)

        # Validate_step
        with torch.no_grad():
            model, valid_loss, error, accuracy = test_evaluation(validation_data, model, criterion)
            valid_losses.append(valid_loss)
            test_error.append(error)

        # Update best accuracy
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), os.path.join(RESULTS, model_name))


        if epoch % print_every == (print_every - 1):
            print(
                f"{datetime.now().time().replace(microsecond=0)} --- "
                f"Epoch: {epoch}\t"
                f"Train loss: {train_loss:.4f}\t"
                f"Valid loss: {valid_loss:.4f}\t"
                f"Test error: {error:.2f}%\t"
                f"Accuracy: {accuracy:.2f}%\t"
            )

    # Save results and plot figures
    np.savetxt(os.path.join(RESULTS, "Test_error.csv"), test_error, delimiter=",")
    np.savetxt(os.path.join(RESULTS, "Train_Losses.csv"), train_losses, delimiter=",")
    np.savetxt(os.path.join(RESULTS, "Valid_Losses.csv"), valid_losses, delimiter=",")
    plot_results(train_losses, valid_losses, test_error, RESULTS)

    return model, optimizer, (train_losses, valid_losses, test_error)


def plot_results(train_losses, valid_losses, test_error, RESULTS):
    """Plot results.

    Args:
        train_losses (List): training losses as calculated in the training_loop
        valid_losses (List): validation losses as calculated in the training_loop
        test_error (List): test error as calculated in the training_loop
    """
    fig = plt.plot(train_losses, "r-s", valid_losses, "b-o")
    plt.title("aihwkit LeNet5")
    plt.legend(fig[:2], ["Training Losses", "Validation Losses"])
    plt.xlabel("Epoch number")
    plt.ylabel("Loss [A.U.]")
    plt.grid(which="both", linestyle="--")
    plt.savefig(os.path.join(RESULTS, "test_losses.png"))
    plt.close()

    fig = plt.plot(test_error, "r-s")
    plt.title("aihwkit LeNet5")
    plt.legend(fig[:1], ["Validation Error"])
    plt.xlabel("Epoch number")
    plt.ylabel("Test Error [%]")
    plt.yscale("log")
    plt.ylim((5e-1, 1e2))
    plt.grid(which="both", linestyle="--")
    plt.savefig(os.path.join(RESULTS, "test_error.png"))
    plt.close()

def unquantized_main():
    # Make sure the directory where to save the results exist.
    # Results include: Loss vs Epoch graph, Accuracy vs Epoch graph and vector data.
    RESULTS = os.path.join(cur_dir, "lenet5_results")
    print("Path to store results: ", RESULTS)
    os.makedirs(RESULTS, exist_ok=True)
    torch.manual_seed(SEED)

    # Load datasets.
    train_data, validation_data = load_images()

    if USE_ANALOG_TRAINING:
        RPU_CONFIG = SingleRPUConfig(device=ConstantStepDevice())
    else:
        RPU_CONFIG = FloatingPointRPUConfig(device=FloatingPointDevice())

    # Prepare the model.
    model = create_analog_network(RPU_CONFIG)
    if USE_CUDA:
        model.cuda()

    print(model)

    print(f"\n{datetime.now().time().replace(microsecond=0)} --- " f"Started LeNet5 Example")

    optimizer = create_sgd_optimizer(model, LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()

    model, optimizer, _ = training_loop(
        model, criterion, optimizer, train_data, validation_data, N_EPOCHS, RESULTS, "LeNet5.th"
    )

    print(f"{datetime.now().time().replace(microsecond=0)} --- " f"Completed LeNet5 Example")

def quantized_main(level: int):
    # Make sure the directory where to save the results exist.
    # Results include: Loss vs Epoch graph, Accuracy vs Epoch graph and vector data.
    RESULTS = os.path.join(cur_dir, "lenet5_quantized_{}__results".format(level))
    print("Path to store results: ", RESULTS)
    os.makedirs(RESULTS, exist_ok=True)
    torch.manual_seed(SEED)

    if USE_ANALOG_TRAINING:
        RPU_CONFIG = SingleRPUConfig(device=ConstantStepDevice())
    else:
        RPU_CONFIG = FloatingPointRPUConfig(device=FloatingPointDevice())
    # add quantization
    RPU_CONFIG.quantization = WeightQuantizerParameter(
            levels = level,
            method = "max",
            use_forward=True,
        )

    # Load datasets.
    train_data, validation_data = load_images()

    # Prepare the model.
    model = create_analog_network(RPU_CONFIG)
    if USE_CUDA:
        model.cuda()

    print(model)

    print(f"\n{datetime.now().time().replace(microsecond=0)} --- " f"Started LeNet5 Example")

    optimizer = create_sgd_optimizer(model, LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()

    model, optimizer, _ = training_loop(
        model, criterion, optimizer, train_data, validation_data, N_EPOCHS, RESULTS, "LeNet5_quantized_{}.th".format(level)
    )

    print(f"{datetime.now().time().replace(microsecond=0)} --- " f"Completed LeNet5 Example")


In [7]:
unquantized_main()

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/lenet5_results
AnalogSequential(
  (0): AnalogConv2d(
    1, 16, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(16,25))
  )
  (1): Tanh()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): AnalogConv2d(
    16, 32, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(32,400))
  )
  (4): Tanh()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Tanh()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): AnalogLinear(
    in_features=512, out_features=128, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(128,512))
  )
  (9): Tanh()
  (10): AnalogLinear(
    in_features=128, out_features=10, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple

In [8]:
quantized_main(3)

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/lenet5_quantized_3__results
AnalogSequential(
  (0): AnalogConv2d(
    1, 16, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(16,25))
  )
  (1): Tanh()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): AnalogConv2d(
    16, 32, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(32,400))
  )
  (4): Tanh()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Tanh()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): AnalogLinear(
    in_features=512, out_features=128, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(128,512))
  )
  (9): Tanh()
  (10): AnalogLinear(
    in_features=128, out_features=10, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(

In [9]:
quantized_main(5)

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/lenet5_quantized_5__results
AnalogSequential(
  (0): AnalogConv2d(
    1, 16, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(16,25))
  )
  (1): Tanh()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): AnalogConv2d(
    16, 32, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(32,400))
  )
  (4): Tanh()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Tanh()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): AnalogLinear(
    in_features=512, out_features=128, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(128,512))
  )
  (9): Tanh()
  (10): AnalogLinear(
    in_features=128, out_features=10, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(

In [10]:
quantized_main(9)

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/lenet5_quantized_9__results
AnalogSequential(
  (0): AnalogConv2d(
    1, 16, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(16,25))
  )
  (1): Tanh()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): AnalogConv2d(
    16, 32, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(32,400))
  )
  (4): Tanh()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Tanh()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): AnalogLinear(
    in_features=512, out_features=128, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(128,512))
  )
  (9): Tanh()
  (10): AnalogLinear(
    in_features=128, out_features=10, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(

In [11]:
quantized_main(17)

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/lenet5_quantized_17__results
AnalogSequential(
  (0): AnalogConv2d(
    1, 16, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(16,25))
  )
  (1): Tanh()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): AnalogConv2d(
    16, 32, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(32,400))
  )
  (4): Tanh()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Tanh()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): AnalogLinear(
    in_features=512, out_features=128, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(128,512))
  )
  (9): Tanh()
  (10): AnalogLinear(
    in_features=128, out_features=10, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile

In [12]:
quantized_main(33)

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/lenet5_quantized_33__results
AnalogSequential(
  (0): AnalogConv2d(
    1, 16, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(16,25))
  )
  (1): Tanh()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): AnalogConv2d(
    16, 32, kernel_size=(5, 5), stride=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(32,400))
  )
  (4): Tanh()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Tanh()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): AnalogLinear(
    in_features=512, out_features=128, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(128,512))
  )
  (9): Tanh()
  (10): AnalogLinear(
    in_features=128, out_features=10, bias=True, FloatingPointRPUConfig
    (analog_module): FloatingPointTile

In [15]:
from aihwkit.nn.conversion import convert_to_analog
src = os.getcwd()
src = os.path.abspath(os.path.join(src, '../src'))
sys.path.append(src)
import plotting as pl


_, validation_data = load_images()

criterion = nn.CrossEntropyLoss()

# For the instantiation, the RPU config object should be OK to be the same for
# the two models
rpu_config = FloatingPointRPUConfig(device=FloatingPointDevice())

# Standard model
RESULTS_NON_QUANTIZED = cur_dir + "/lenet5_results"
state_dict = torch.load(os.path.join(RESULTS_NON_QUANTIZED, "LeNet5.th"), map_location=DEVICE)
non_quantized_model = create_analog_network(rpu_config).to(DEVICE)
non_quantized_model.load_state_dict(state_dict, strict=True, load_rpu_config=False)

non_quantized_model.eval()

_, _ , _, accuracy_non_quantized = test_evaluation(validation_data, non_quantized_model, criterion)
print("Accuracy non quantized: ", accuracy_non_quantized)

# Plot the weights through the layers
pl.generate_moving_hist(non_quantized_model, title = "NON-QUANTIZED model", file_name = RESULTS_NON_QUANTIZED + "/weights_hist_non_quantized.gif", range = (-1.5, 1.5), top = None, split_by_rows=False)


accuracy_quantized = {}
for level in [3, 5, 9, 17, 33]:
    # Quantized model
    RESULTS_QUANTIZED = cur_dir + "/lenet5_quantized_{}__results".format(level)
    state_dict = torch.load(os.path.join(RESULTS_QUANTIZED, "LeNet5_quantized_{}.th".format(level)), map_location=DEVICE)
    quantized_model = create_analog_network(rpu_config).to(DEVICE)
    quantized_model.load_state_dict(state_dict, strict=True, load_rpu_config=False)


    rpu_config.quantization = WeightQuantizerParameter(
            levels = level,
            method = "max"
        )
    
    quantized_model = convert_to_analog(quantized_model, rpu_config)
    quantized_model.eval()

    _, _, _, accuracy_quantized[level] = test_evaluation(validation_data, quantized_model, criterion)
    print("Accuracy quantized {}: ".format(level), accuracy_quantized[level])

    # Plot the weights through the layers
    pl.generate_moving_hist(quantized_model, title = "QUANTIZED {} model".format(level), file_name = RESULTS_QUANTIZED + "/weights_hist_quantized_{}.gif".format(level), range = (-1.5, 1.5), top = None, split_by_rows=False)


Accuracy non quantized:  98.82
Accuracy quantized 3:  98.29
Accuracy quantized 5:  98.75
Accuracy quantized 9:  98.8
Accuracy quantized 17:  98.76
Accuracy quantized 33:  98.76


<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>